In [10]:
# === Cell 1: Install Dependencies (Gradio/FastAPI/Starlette fixed) ===
# Run ONCE, then restart the kernel.

import os
import glob
import shutil

# Clean broken leftover ~ttpx package folders
site = os.path.expanduser(r"~\AppData\Roaming\Python\Python313\site-packages")
for path in glob.glob(os.path.join(site, "~ttpx*")):
    print("Deleting broken package metadata:", path)
    shutil.rmtree(path, ignore_errors=True)

print("✅ Cleanup step complete.")

%pip install --upgrade pip --user

# Remove conflicting web-stack packages first
%pip uninstall -y gradio gradio_client fastapi starlette jinja2 huggingface_hub transformers tokenizers markupsafe pillow websockets httpx openai

# Install exact versions for this notebook
%pip install --user --force-reinstall "audioop-lts"
%pip install --user --force-reinstall "huggingface_hub==0.25.2"
%pip install --user --force-reinstall "gradio==4.36.1"
%pip install --user --force-reinstall "gradio-client==1.0.1"
%pip install --user --force-reinstall "fastapi==0.112.2"
%pip install --user --force-reinstall "starlette==0.37.2"
%pip install --user --force-reinstall "jinja2==3.1.4"
%pip install --user --force-reinstall "transformers==4.48.0"
%pip install --user --force-reinstall "tokenizers==0.21.4"
%pip install --user --force-reinstall "markupsafe==2.1.5"
%pip install --user --force-reinstall "pillow==10.4.0"
%pip install --user --force-reinstall "websockets==11.0.3"

# Core libs
%pip install --user --force-reinstall "numpy<2.0" "pandas==2.2.3" timm requests

# PyTorch CPU
%pip install --user --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

# LM Studio wrapper
%pip install --user --force-reinstall "langchain-openai>=0.1.23"

print("✅ Final install complete.")
print("➡️ Restart the kernel now.")

In [1]:
import gradio, fastapi, starlette, jinja2, transformers, huggingface_hub
print("gradio:", gradio.__version__)
print("fastapi:", fastapi.__version__)
print("starlette:", starlette.__version__)
print("jinja2:", jinja2.__version__)
print("transformers:", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)

gradio: 4.36.1
fastapi: 0.112.2
starlette: 0.37.2
jinja2: 3.1.6
transformers: 4.48.0
huggingface_hub: 0.36.2


In [2]:
# === Cell 2: Config for LM Studio ===
import os
import json
import requests

# Point to LM Studio's local OpenAI-compatible API
os.environ["OPENAI_BASE_URL"] = "http://127.0.0.1:1234/v1"

# LM Studio only needs a non-empty API key string
os.environ["OPENAI_API_KEY"] = "lmstudio-key"

# Model name must match what LM Studio exposes
MODEL = "mistral-7b-instruct-v0.2"
os.environ["MODEL"] = MODEL

print("✅ Config loaded:")
print("  OPENAI_BASE_URL:", os.environ["OPENAI_BASE_URL"])
print("  OPENAI_API_KEY :", "(set)")
print("  MODEL          :", MODEL)

# Quick connectivity check
try:
    resp = requests.get(
        f"{os.environ['OPENAI_BASE_URL'].rstrip('/')}/models",
        headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"},
        timeout=6
    )
    resp.raise_for_status()
    models = resp.json()
    print("\n✅ LM Studio is reachable. Models endpoint responded.")
    print(json.dumps(models, indent=2)[:800], "...\n")
except Exception as e:
    print("\n⚠️ Could not reach LM Studio.")
    print("   • Is LM Studio running?")
    print("   • Is the API server ON in LM Studio?")
    print("   • Is a chat model loaded?")
    print("Error:", repr(e))

✅ Config loaded:
  OPENAI_BASE_URL: http://127.0.0.1:1234/v1
  OPENAI_API_KEY : (set)
  MODEL          : mistral-7b-instruct-v0.2

✅ LM Studio is reachable. Models endpoint responded.
{
  "data": [
    {
      "id": "mistral-7b-instruct-v0.2",
      "object": "model",
      "owned_by": "organization_owner"
    },
    {
      "id": "text-embedding-nomic-embed-text-v1.5",
      "object": "model",
      "owned_by": "organization_owner"
    },
    {
      "id": "rag-qwen2.5-7b",
      "object": "model",
      "owned_by": "organization_owner"
    },
    {
      "id": "openai/gpt-oss-20b",
      "object": "model",
      "owned_by": "organization_owner"
    }
  ],
  "object": "list"
} ...



In [4]:
# === Cell 3: Core imports, a tiny nutrition DB, and helper utilities ===
from typing import List, Dict, Any
from PIL import Image
import numpy as np
import pandas as pd

# --- A tiny nutrition database (per 100g). Great for teaching; easy to extend. ---
NUTRITION_DB = {
    "salmon": {"calories": 208, "protein_g": 20, "fat_g": 13, "carbs_g": 0, "fiber_g": 0},
    "asparagus": {"calories": 20, "protein_g": 2.2, "fat_g": 0.1, "carbs_g": 3.9, "fiber_g": 2.1},
    "tomato": {"calories": 18, "protein_g": 0.9, "fat_g": 0.2, "carbs_g": 3.9, "fiber_g": 1.2},
    "rice": {"calories": 130, "protein_g": 2.7, "fat_g": 0.3, "carbs_g": 28, "fiber_g": 0.4},
    "pasta": {"calories": 131, "protein_g": 5, "fat_g": 1.1, "carbs_g": 25, "fiber_g": 1.3},
    "avocado": {"calories": 160, "protein_g": 2, "fat_g": 15, "carbs_g": 9, "fiber_g": 7},
    "egg": {"calories": 155, "protein_g": 13, "fat_g": 11, "carbs_g": 1.1, "fiber_g": 0},
    "chicken_breast": {"calories": 165, "protein_g": 31, "fat_g": 3.6, "carbs_g": 0, "fiber_g": 0},
    "beef_steak": {"calories": 271, "protein_g": 25, "fat_g": 19, "carbs_g": 0, "fiber_g": 0},
    "broccoli": {"calories": 34, "protein_g": 2.8, "fat_g": 0.4, "carbs_g": 7, "fiber_g": 2.6},
    "potato": {"calories": 77, "protein_g": 2, "fat_g": 0.1, "carbs_g": 17, "fiber_g": 2.2},
    "spinach": {"calories": 23, "protein_g": 2.9, "fat_g": 0.4, "carbs_g": 3.6, "fiber_g": 2.2},
    "lemon": {"calories": 29, "protein_g": 1.1, "fat_g": 0.3, "carbs_g": 9.3, "fiber_g": 2.8},
    "olive_oil": {"calories": 884, "protein_g": 0, "fat_g": 100, "carbs_g": 0, "fiber_g": 0},
}

# Candidate labels for zero-shot recognition
CANDIDATE_LABELS = [
    "salmon", "asparagus", "tomato", "rice", "pasta", "avocado", "egg",
    "chicken breast", "beef steak", "broccoli", "potato", "spinach", "lemon",
    "olive oil", "cheeseburger", "pizza", "pancakes", "sushi", "tofu"
]

def to_safe_key(label: str) -> str:
    """Normalize model labels to our DB keys (simple mapping demo)."""
    label = label.lower().strip().replace(" ", "_")
    mapping = {
        "chicken_breast": "chicken_breast",
        "chicken": "chicken_breast",
        "beef_steak": "beef_steak",
        "steak": "beef_steak",
    }
    return mapping.get(label, label)

def scale_nutrition(per100g: Dict[str, float], grams: float) -> Dict[str, float]:
    """Scale per-100g nutrition to any gram amount."""
    factor = grams / 100.0
    return {k: round(v * factor, 2) for k, v in per100g.items()}

In [5]:
# === Cell 4: Load CLIP + zero-shot label ranking ===
# We pick CLIP because it's small, fast to download, and works well for label matching.

import torch
from transformers import CLIPProcessor, CLIPModel

DEVICE = "cpu"  # set to "cuda" if you installed GPU torch and have a GPU
CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"

clip_model = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(DEVICE)
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)

def clip_rank_labels(pil_image: Image.Image, candidate_texts: List[str], top_k: int = 3):
    """
    Compare an image against candidate labels and return the top_k by probability.
    This is true zero-shot classification using CLIP's image-text similarity.
    """
    inputs = clip_processor(
        text=[f"a photo of {t}" for t in candidate_texts],
        images=pil_image,
        return_tensors="pt",
        padding=True
    ).to(DEVICE)

    with torch.no_grad():
        outputs = clip_model(**inputs)
        logits_per_image = outputs.logits_per_image  # shape (1, num_texts)
        probs = logits_per_image.softmax(dim=1).cpu().numpy().flatten()

    ranked = sorted(zip(candidate_texts, probs), key=lambda x: x[1], reverse=True)[:top_k]
    return [{"label": lab, "prob": float(p)} for lab, p in ranked]

In [6]:
print("CLIP model loaded:", clip_model is not None)
print("CLIP processor loaded:", clip_processor is not None)
print("Device:", DEVICE)

CLIP model loaded: True
CLIP processor loaded: True
Device: cpu


In [7]:
# === Cell 5: LLM + Agent Roles (fixed for LM Studio + Mistral) ===
import os
from langchain_openai import ChatOpenAI

# --- Create a LangChain chat LLM backed by LM Studio ---
chat_llm = ChatOpenAI(
    model=os.getenv("MODEL", "mistral-7b-instruct-v0.2"),   # must match LM Studio's model name
    api_key=os.getenv("OPENAI_API_KEY", "lmstudio-key"),    # any non-empty string
    base_url=os.getenv("OPENAI_BASE_URL", "http://127.0.0.1:1234/v1"),
    temperature=0.3,
    max_tokens=700,
)

# --- Utility: call the chat LLM with a system + user prompt and return content text ---
def llm_chat(system_prompt: str, user_prompt: str) -> str:
    """
    LM Studio compatibility: many local prompt templates only accept a single user message.
    So we fold the 'system' guidance into one user message.
    """
    one_message = (
        "### System instruction (follow strictly):\n"
        f"{system_prompt}\n\n"
        "### User request:\n"
        f"{user_prompt}"
    )
    msg = chat_llm.invoke(one_message)
    return getattr(msg, "content", str(msg))

# ---- Define agent roles (lightweight, avoids CrewAI validation issues in this environment) ----
vision_analyst = {
    "role": "Vision Analyst",
    "goal": "Identify likely foods in a photo using a vision tool and report top-3 with confidence.",
    "backstory": "A careful computer-vision assistant focused on accurate label suggestions."
}

nutritionist = {
    "role": "Nutritionist",
    "goal": "Estimate calories/macros from ingredient list + portions using a small nutrition DB.",
    "backstory": "A data-driven nutrition expert who is transparent about approximations."
}

recipe_planner = {
    "role": "Recipe Planner",
    "goal": "Create a simple, tasty, personalized recipe that matches dietary goals.",
    "backstory": "A friendly home chef who writes clear steps and practical substitutions."
}

print("✅ Cell 5 loaded successfully.")
print("LLM model:", os.getenv("MODEL", "mistral-7b-instruct-v0.2"))
print("Agent roles:", [vision_analyst["role"], nutritionist["role"], recipe_planner["role"]])

✅ Cell 5 loaded successfully.
LLM model: mistral-7b-instruct-v0.2
Agent roles: ['Vision Analyst', 'Nutritionist', 'Recipe Planner']


In [8]:
# === Cell 6: Tools + coordinator ===

def tool_detect_foods(image: Image.Image, top_k: int = 3) -> List[Dict[str, Any]]:
    """Use CLIP zero-shot to propose top_k food labels with probabilities."""
    return clip_rank_labels(image, CANDIDATE_LABELS, top_k=top_k)

def tool_estimate_nutrition(detections: List[Dict[str, Any]], grams_map: Dict[str, float]) -> Dict[str, Any]:
    """
    Convert detected labels + gram estimates into per-ingredient + total macro counts.
    grams_map keys can be 'salmon', 'beef steak', etc.; we normalize to DB keys.
    """
    items = []
    totals = {"calories": 0, "protein_g": 0, "fat_g": 0, "carbs_g": 0, "fiber_g": 0}

    for d in detections:
        raw_label = d["label"]
        key = to_safe_key(raw_label)
        grams = float(grams_map.get(raw_label, grams_map.get(key, 0)))

        if grams <= 0:
            continue
        if key not in NUTRITION_DB:
            continue

        scaled = scale_nutrition(NUTRITION_DB[key], grams)
        items.append({"ingredient": raw_label, "grams": grams, **scaled})

        for k in totals:
            totals[k] += scaled[k]

    totals = {k: round(v, 2) for k, v in totals.items()}
    return {"items": items, "totals": totals}

def coordinator(
    image: Image.Image,
    user_prefs: str,
    servings: int = 2,
    calories_goal: int = 600,
    grams_overrides: Dict[str, float] | None = None
) -> Dict[str, Any]:
    """
    High-level pipeline:
      1) Vision Analyst -> detect labels
      2) Nutritionist   -> estimate macros
      3) Recipe Planner -> generate recipe text
    """
    # 1) Vision
    detections = tool_detect_foods(image, top_k=3)
    labels = [d["label"] for d in detections]

    # 2) Portion defaults
    default_grams = {
        lab: 150.0 if any(k in lab for k in ["salmon", "steak", "chicken"]) else 80.0
        for lab in labels
    }
    grams_map = default_grams | (grams_overrides or {})

    # 3) Nutrition
    nutrition = tool_estimate_nutrition(detections, grams_map)

    # 4) Recipe (LLM)
    system = "You are a professional chef and nutrition coach. Be concise, friendly, and practical."
    user = f"""
Detected ingredients (top-3): {labels}
Estimated totals (approx, for the whole dish): {nutrition['totals']}
User preferences/restrictions: {user_prefs}
Target calories per serving: ~{calories_goal}
Servings: {servings}

Write:
1) A brief 2-sentence assessment of the plate and nutrition.
2) A personalized recipe (ingredients list in grams + step-by-step instructions).
3) One tip to adjust calories up or down while keeping protein reasonable.
"""
    recipe_text = llm_chat(system, user)

    formatted_detections = [
        {"label": d["label"], "probability": round(d["prob"], 4)}
        for d in detections
    ]

    return {
        "detections": formatted_detections,
        "nutrition": nutrition,
        "recipe_text": recipe_text.strip()
    }

print("✅ Cell 6 loaded successfully.")

✅ Cell 6 loaded successfully.


In [9]:
# === Cell 7: Quick direct test (recommended before UI) ===
from PIL import Image
import numpy as np

# Create a dummy image just to test the full pipeline wiring
dummy_img = Image.fromarray(np.zeros((224, 224, 3), dtype=np.uint8)).convert("RGB")

result = coordinator(
    dummy_img,
    user_prefs="high protein, no dairy",
    servings=2,
    calories_goal=600
)

print("Detections:")
print(result["detections"])

print("\nNutrition:")
print(result["nutrition"])

print("\nRecipe:")
print(result["recipe_text"][:1000])  # print first part so output isn't huge

Detections:
[{'label': 'olive oil', 'probability': 0.2402}, {'label': 'pizza', 'probability': 0.0834}, {'label': 'rice', 'probability': 0.0664}]

Nutrition:
{'items': [{'ingredient': 'olive oil', 'grams': 80.0, 'calories': 707.2, 'protein_g': 0.0, 'fat_g': 80.0, 'carbs_g': 0.0, 'fiber_g': 0.0}, {'ingredient': 'rice', 'grams': 80.0, 'calories': 104.0, 'protein_g': 2.16, 'fat_g': 0.24, 'carbs_g': 22.4, 'fiber_g': 0.32}], 'totals': {'calories': 811.2, 'protein_g': 2.16, 'fat_g': 80.24, 'carbs_g': 22.4, 'fiber_g': 0.32}}

Recipe:
1) The detected ingredients, pizza and rice, provide a significant amount of calories with limited protein and fiber. This dish exceeds the target calorie count and does not meet your high protein and no dairy preferences.
2) To create a more balanced meal that fits your dietary needs, consider the following recipe for a Grilled Chicken and Quinoa Bowl:
   - Ingredients (per serving): 150g cooked quinoa, 150g grilled chicken breast, 30g sliced almonds, 1 medium av

In [10]:
# === Cell 8: Gradio UI ===

import sys
import types

# Patch audio modules for Python 3.13 / Windows if needed
sys.modules["audioop"] = types.ModuleType("audioop")
sys.modules["pyaudioop"] = types.ModuleType("pyaudioop")

import json
import gradio as gr
import pandas as pd
from PIL import Image
from typing import Any, Dict

def nutrition_df(nutrition: Dict[str, Any]) -> pd.DataFrame:
    """Render nutrition dict as a DataFrame with a TOTAL row."""
    items = nutrition.get("items", [])
    totals = nutrition.get("totals", {})

    if not items:
        return pd.DataFrame([totals]) if totals else pd.DataFrame()

    df = pd.DataFrame(items)
    total_row = {
        "ingredient": "TOTAL",
        "grams": float(df["grams"].sum()) if "grams" in df.columns else 0,
        **totals
    }
    return pd.concat([df, pd.DataFrame([total_row])], ignore_index=True)

def run_pipeline(image, servings, calories_goal, dietary_prefs, grams_overrides_json):
    """Gradio callback for the full pipeline."""
    if image is None:
        return (
            {"error": "Please upload an image first."},
            pd.DataFrame(),
            "Please upload a food photo to analyze."
        )

    pil = Image.fromarray(image).convert("RGB")

    overrides = {}
    if grams_overrides_json and grams_overrides_json.strip():
        try:
            overrides = json.loads(grams_overrides_json)
        except Exception:
            return (
                {"error": "Invalid JSON in portion overrides."},
                pd.DataFrame(),
                'Use valid JSON like: {"salmon": 180, "asparagus": 90}'
            )

    try:
        result = coordinator(
            pil,
            user_prefs=dietary_prefs or "",
            servings=int(servings),
            calories_goal=int(calories_goal),
            grams_overrides=overrides or None
        )
    except Exception as e:
        return (
            {"error": "Pipeline failed."},
            pd.DataFrame(),
            f"An error occurred while running the pipeline:\n\n{str(e)}"
        )

    labels = result.get("detections", [])
    df = nutrition_df(result.get("nutrition", {}))
    recipe = result.get("recipe_text", "No recipe generated.")

    return labels, df, recipe

with gr.Blocks() as demo:
    gr.Markdown("# 🍽️ Agentic Food Analyzer")
    gr.Markdown("Upload a food photo → get nutrition estimates → receive a personalized recipe.")

    with gr.Row():
        with gr.Column():
            image_in = gr.Image(type="numpy", label="Upload a food photo")
            servings_in = gr.Slider(1, 6, value=2, step=1, label="Servings")
            calories_goal_in = gr.Slider(200, 1200, value=600, step=50, label="Target Calories per Serving")
            dietary_prefs_in = gr.Textbox(
                label="Dietary preferences / restrictions",
                placeholder="Example: high protein, no dairy, gluten-free"
            )
            grams_overrides_in = gr.Textbox(
                label="Optional portion overrides as JSON",
                placeholder='{"salmon": 180, "asparagus": 90}'
            )
            run_btn = gr.Button("Analyze & Create Recipe")

        with gr.Column():
            labels_out = gr.JSON(label="Top-3 detected foods")
            nutrition_out = gr.Dataframe(label="Estimated Nutrition")
            recipe_out = gr.Textbox(label="Personalized Recipe", lines=18)

    run_btn.click(
        fn=run_pipeline,
        inputs=[image_in, servings_in, calories_goal_in, dietary_prefs_in, grams_overrides_in],
        outputs=[labels_out, nutrition_out, recipe_out]
    )

demo.launch(share=True)

Running on local URL:  http://127.0.0.1:7860
IMPORTANT: You are using gradio version 4.36.1, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://a51bdf7e90781b5520.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "<python site-packages>", line 420, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        self.scope, self.receive, self.send
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "<python site-packages>", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<python site-packages>", line 1054, in __call__
    await super().__call__(scope, receive, send)
  File "<python site-packages>", line 123, in __call__
    await self.middleware_stack(scope, receive, send)
  File "<python site-packages>", line 186, in __call__
    raise exc
  File "<python site-packages>", line 164, in __call__
    await self.app(scope, receive, _send)
  File "<python site-packages>", line 714, in __call__
    await self.app(scope, receive, send)
  File "<python site-pac